# 04 - Preprocessing: outliers and missing values

**Goal:** decide how to treat outliers and missing values. The decisions are applied later inside the sklearn `Pipeline`, fitted on train only.

- Uses the clean data saved by `01_data_cleaning.ipynb` (run it first).
- The clear errors (`1e9`, sign flips, inflated values) were already fixed in 01. Here the question is whether the remaining extreme values are real, and how much weight they should have.
- Sections: 1. Outliers in the features, 2. Outliers in the target. Missing values come next.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data"
TARGET, SPEND = "repeat_purchase_90d", "spend_next_90d"
BLUE, ORANGE, GREY, RED = "#2f6690", "#d9822b", "#9aa5b1", "#c0392b"

# Same plot style as the other notebooks
pio.templates["eda"] = go.layout.Template(layout=dict(
    font=dict(size=13), title=dict(font=dict(size=17)), height=450,
    margin=dict(t=80, l=60, r=30, b=50), hoverlabel=dict(bgcolor="white")))
pio.templates.default = "plotly_white+eda"

# Clean data saved at the end of notebook 01 (categorical labels are numbers stored as text)
CAT_COLS = ["payment_type", "salesperson", "commercial_zone", "transport_method", "warehouse", "document_series"]
df = pd.read_csv(f"{DATA_DIR}/processed/train_clean.csv", parse_dates=["purchase_date"], dtype={c: str for c in CAT_COLS})
num_cols = df.select_dtypes("number").columns.drop(["ID", "customer_id", TARGET, SPEND, "random_noise"], errors="ignore")  # random_noise is not a feature
print("train:", df.shape, "| numeric features:", len(num_cols), "| negative values:", (df[num_cols] < 0).sum().sum())

## 1. Outliers in the features

Two questions for each numeric column: **how many values are extreme**, and **are they real or errors**?

First the distribution of each column (choose it in the dropdown), then the number of rows outside the IQR limits: below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR (the box-plot rule).

In [ ]:
fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3], vertical_spacing=0.15,
                    subplot_titles=("Histogram of the middle 98% (1%-99%)", "Box plot of all values"))
buttons = []
for i, variable in enumerate(num_cols):
    values = df[variable].dropna()
    low, high = values.quantile([0.01, 0.99])
    bulk = values[values.between(low, high)]
    fig.add_histogram(x=bulk, nbinsx=50, marker_color=BLUE, name=variable, visible=(i == 0), row=1, col=1)
    fig.add_box(x=values, marker_color=BLUE, boxpoints="outliers", name="", visible=(i == 0), row=2, col=1)

    title = (f"{variable}<br><sup>median {values.median():,.2f} | max {values.max():,.2f} | "
             f"missing {df[variable].isna().sum()} | outside 1%-99%: {len(values) - len(bulk)}</sup>")
    visible = [False] * (2 * len(num_cols))
    visible[2 * i] = visible[2 * i + 1] = True
    buttons.append(dict(label=variable, method="update", args=[{"visible": visible}, {"title.text": title}]))

fig.update_layout(title=buttons[0]["args"][1]["title.text"], height=600, showlegend=False,
                  updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.2, yanchor="top")])
fig.update_yaxes(title_text="number of rows", row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.show()

In [ ]:
# Rows outside the IQR limits, per column
rows = {}
for col in num_cols:
    values = df[col].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    below = (values < q1 - 1.5 * iqr).sum()
    above = (values > q3 + 1.5 * iqr).sum()
    rows[col] = {"below lower limit": below, "above upper limit": above,
                 "% outliers": round(100 * (below + above) / len(values), 1)}
iqr_table = pd.DataFrame(rows).T.astype({"below lower limit": int, "above upper limit": int})
iqr_table.sort_values("% outliers", ascending=False)

### Key insight - Section 1

**Found**
- **All outliers are on the high side:** no column has a value below the lower limit. The IQR rule flags 5-16% of almost every column, which is the long right tail of money and count columns, not errors.
- In almost constant columns (`Total Portes` and `Total Desc_ Global` are mostly 0, `invoice_count_current_purchase` is mostly 1) the IQR is 0, so every other value counts as an outlier (e.g. all 1,053 purchases with shipping). For them the useful information is "usual value or not".
- **The extreme values are real customers, not errors.** In notebook 01 the history columns were checked against the purchase dates, and the impossible values (including 3 `prior_max_purchase_value` about 10 times too large) were fixed there.

**For preprocessing**
- Don't delete rows: the extremes are real customers, and test has even more loyal, bigger customers (notebook 02, Section 8).
- **Tree models** (Random Forest, gradient boosting): no treatment needed, they only use the order of the values.
- **Distance- and weight-based models** (logistic/linear regression, KNN, SVM, neural networks): apply `log1p` to the money and count columns, then scale. Optionally also cap at the 1st/99th percentile, with the limits fitted on train inside the pipeline. Compare both in validation.

## 2. Outliers in the target `spend_next_90d`

The regression is evaluated with **RMSE, MAE and R²** on the customers who actually returned (project brief). RMSE squares each error, so a few very large values can dominate it.

In [ ]:
returned = df[df[SPEND].notna()]
spend = returned[SPEND]
top = spend >= spend.quantile(0.99)
squared_dev = (spend - spend.mean()) ** 2

print(spend.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())
print(f"\nTop 1%: {top.sum()} rows from {returned.loc[top, 'customer_id'].nunique()} customers")
print(f"  share of all spend:               {spend[top].sum() / spend.sum():.1%}")
print(f"  share of the squared deviations:  {squared_dev[top].sum() / squared_dev.sum():.1%}  (what RMSE is built on)")
print(f"  median spend_365d, top 1% vs rest: {returned.loc[top, 'spend_365d'].median():,.0f} vs "
      f"{returned.loc[~top, 'spend_365d'].median():,.0f}")

In [ ]:
fig = go.Figure()
fig.add_scatter(x=np.log1p(returned.loc[~top, "spend_365d"]), y=np.log1p(spend[~top]), mode="markers",
                marker=dict(size=5, color=BLUE, opacity=0.3), name="other returners")
fig.add_scatter(x=np.log1p(returned.loc[top, "spend_365d"]), y=np.log1p(spend[top]), mode="markers",
                marker=dict(size=9, color=RED), name="top 1% spend")
fig.update_layout(title="The biggest spenders were already big spenders", height=550,
                  xaxis_title="log(1 + spend_365d)", yaxis_title="log(1 + spend_next_90d)")
fig.show()

In [ ]:
# Three ways to treat the target, compared by their skewness
p99 = spend.quantile(0.99)
pd.DataFrame({"skewness": [spend.skew(), np.log1p(spend).skew(), spend.clip(upper=p99).skew()],
              "values changed": [0, 0, (spend > p99).sum()]},
             index=["raw", "log1p", f"capped at the 99th pct ({p99:,.0f})"]).round(2)

### Key insight - Section 2

**Found**
- Half of the returners spend less than about 940 EUR, but the maximum is about 25,600 EUR.
- **The top 1% (36 rows) come from only 5 customers.** They hold 9% of all spend but **33% of the squared deviations**, so they will weigh heavily on RMSE.
- They are real and predictable: their spend in the past year is about 8 times the others' (median 33,700 vs 4,300 EUR), and the spend target was already confirmed from the purchase history in notebook 01.
- `log1p` makes the target almost symmetric (skewness 3.1 -> -0.6). Capping at the 99th percentile still leaves it skewed (2.2).

**For preprocessing**
- **Keep the big spenders, and don't cap the target.** Test is scored on the real values, and these are the customers the business most wants to prioritise. A capped model would under-predict them on purpose.
- Train the regression on `log1p(spend)` and on the raw target, and compare them in validation with RMSE, MAE and R² **on the original scale** (convert back with `expm1`). A log model predicts a "typical" value rather than the average, so it can under-predict big spenders. Validation decides.